# Preliminary Modeling: Review Text -> Rating Prediction

**Goal:** Validate that review text content carries signal for rating prediction. This informs feature engineering choices for the two-stage ranking recommendation system.

**Models compared:**
1. Ridge regression on simple features (word count, keywords) — baseline
2. Ridge regression on TF-IDF features — text signal check
3. MLP on TF-IDF (PyTorch) — nonlinear capacity
4. Multiclass classification (5 classes) — MLP + LogisticRegression

**Evaluation:**
- Regression: RMSE, MAE, Rounded Accuracy
- Classification: Accuracy, Macro-F1, Confusion Matrix

In [ ]:
import sys
sys.path.insert(0, "/path/to/R-project")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, accuracy_score,
    f1_score, confusion_matrix, classification_report,
)
from scipy.sparse import issparse
from tqdm import tqdm

from scripts.data.loader import load_sampled

CATEGORIES = ["All_Beauty", "Video_Games", "Books", "Electronics"]
SAMPLE_SIZE = {"All_Beauty": None, "Video_Games": 500_000, "Books": 500_000, "Electronics": 500_000}
FIGURES_DIR = Path("/path/to/R-project/figures")
RESULTS_DIR = Path("/path/to/R-project/results")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

## Load and prepare data

In [ ]:
# Load sampled data for each category, keep only needed columns
datasets = {}
for cat in CATEGORIES:
    df = load_sampled(cat, n=SAMPLE_SIZE[cat],
                      columns=["rating", "text", "title", "verified_purchase"])
    # Drop rows with missing text
    df = df.dropna(subset=["text"]).reset_index(drop=True)
    df["text"] = df["text"].astype(str)
    datasets[cat] = df
    print(f"{cat}: {len(df):,} reviews, rating distribution:")
    print(df["rating"].value_counts().sort_index().to_dict())

## Feature Engineering

**Simple features:** word_count, char_count, exclamation_count, question_mark_count, uppercase_ratio, verified_purchase

**TF-IDF features:** `TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=5)`

In [ ]:
def extract_simple_features(df):
    """Extract simple numeric features from review text."""
    feats = pd.DataFrame()
    feats["word_count"] = df["text"].str.split().str.len()
    feats["char_count"] = df["text"].str.len()
    feats["exclamation_count"] = df["text"].str.count("!")
    feats["question_count"] = df["text"].str.count(r"\?")
    feats["uppercase_ratio"] = df["text"].apply(
        lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
    )
    feats["verified_purchase"] = df["verified_purchase"].astype(float)
    return feats.values  # numpy array


def prepare_category(df, tfidf_params=None):
    """Split data and extract features for one category.
    
    Returns dict with train/test splits for simple and TF-IDF features.
    """
    if tfidf_params is None:
        tfidf_params = dict(max_features=10000, ngram_range=(1, 2), min_df=5, 
                            sublinear_tf=True)
    
    # Drop rows whose rating is outside the valid 1-5 range. CrossEntropyLoss
    # would otherwise see label -1 (from rating==0) and crash with IndexError.
    n_before = len(df)
    df = df[df["rating"].between(1, 5)].reset_index(drop=True)
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"  Dropped {n_dropped} rows with rating outside 1-5")
    
    y = df["rating"].values
    y_class = (df["rating"].astype(int) - 1).values  # 0-indexed for classification
    
    # Train/test split (80/20, stratified by rating)
    idx_train, idx_test = train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=42, stratify=y_class
    )
    
    # Simple features
    X_simple = extract_simple_features(df)
    
    # TF-IDF
    tfidf = TfidfVectorizer(**tfidf_params)
    X_tfidf = tfidf.fit_transform(df["text"].iloc[idx_train])
    X_tfidf_test = tfidf.transform(df["text"].iloc[idx_test])
    
    return {
        "X_simple_train": X_simple[idx_train],
        "X_simple_test": X_simple[idx_test],
        "X_tfidf_train": X_tfidf,
        "X_tfidf_test": X_tfidf_test,
        "y_train": y[idx_train],
        "y_test": y[idx_test],
        "y_class_train": y_class[idx_train],
        "y_class_test": y_class[idx_test],
        "tfidf": tfidf,
    }


# Prepare all categories
prepared = {}
for cat in CATEGORIES:
    print(f"Preparing {cat}...")
    prepared[cat] = prepare_category(datasets[cat])
    print(f"  Train: {len(prepared[cat]['y_train']):,}, Test: {len(prepared[cat]['y_test']):,}")
    print(f"  TF-IDF shape: {prepared[cat]['X_tfidf_train'].shape}")

## Model 1: Ridge Regression on Simple Features (Baseline)

In [ ]:
def eval_regression(y_true, y_pred):
    """Evaluate regression: RMSE, MAE, rounded accuracy."""
    y_pred_clipped = np.clip(y_pred, 1, 5)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)
    y_rounded = np.clip(np.round(y_pred_clipped), 1, 5)
    acc = accuracy_score(y_true.astype(int), y_rounded.astype(int))
    return {"rmse": rmse, "mae": mae, "rounded_acc": acc}


# Model 1: Ridge on simple features
model1_results = {}
for cat in CATEGORIES:
    d = prepared[cat]
    model = Ridge(alpha=1.0)
    model.fit(d["X_simple_train"], d["y_train"])
    y_pred = model.predict(d["X_simple_test"])
    metrics = eval_regression(d["y_test"], y_pred)
    metrics["model"] = "Ridge_Simple"
    metrics["category"] = cat
    model1_results[cat] = metrics
    print(f"{cat}: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, "
          f"Rounded Acc={metrics['rounded_acc']:.4f}")

pd.DataFrame(model1_results.values())

## Model 2: Ridge Regression on TF-IDF Features

In [ ]:
# Model 2: Ridge on TF-IDF (sparse matrix — memory efficient)
model2_results = {}
for cat in CATEGORIES:
    d = prepared[cat]
    model = Ridge(alpha=1.0)
    model.fit(d["X_tfidf_train"], d["y_train"])
    y_pred = model.predict(d["X_tfidf_test"])
    metrics = eval_regression(d["y_test"], y_pred)
    metrics["model"] = "Ridge_TFIDF"
    metrics["category"] = cat
    model2_results[cat] = metrics
    print(f"{cat}: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, "
          f"Rounded Acc={metrics['rounded_acc']:.4f}")

pd.DataFrame(model2_results.values())

## Model 3: MLP on TF-IDF (PyTorch, Regression)

Architecture: `10000 -> 512 -> ReLU -> Dropout(0.3) -> 128 -> ReLU -> 1`

Custom Dataset indexes into the sparse TF-IDF matrix and densifies per-batch to avoid materializing the full 10K-wide dense matrix.

In [ ]:
class SparseDataset(Dataset):
    """Dataset that indexes into a sparse matrix, densifying per-sample."""
    def __init__(self, X_sparse, y):
        self.X = X_sparse
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].toarray().squeeze(), dtype=torch.float32)
        return x, self.y[idx]


class RatingMLP(nn.Module):
    """Simple MLP for rating prediction."""
    def __init__(self, input_dim, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim),
        )
    
    def forward(self, x):
        return self.net(x)


def train_mlp(model, train_loader, test_loader, epochs=10, lr=1e-3, 
              criterion=None, task="regression"):
    """Train MLP and return test predictions."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if criterion is None:
        criterion = nn.MSELoss() if task == "regression" else nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            if task == "regression":
                loss = criterion(out.squeeze(), y_batch)
            else:
                loss = criterion(out, y_batch.long())
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
        avg_loss = total_loss / len(train_loader.dataset)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    # Predict on test
    model.eval()
    preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            out = model(X_batch)
            preds.append(out.cpu())
    return torch.cat(preds)

In [ ]:
# Model 3: MLP Regression on TF-IDF
model3_results = {}
for cat in CATEGORIES:
    print(f"\n{'='*40} {cat}")
    d = prepared[cat]
    
    train_ds = SparseDataset(d["X_tfidf_train"], d["y_train"])
    test_ds = SparseDataset(d["X_tfidf_test"], d["y_test"])
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)
    
    input_dim = d["X_tfidf_train"].shape[1]
    model = RatingMLP(input_dim, output_dim=1)
    preds = train_mlp(model, train_loader, test_loader, epochs=10, task="regression")
    
    y_pred = preds.squeeze().numpy()
    metrics = eval_regression(d["y_test"], y_pred)
    metrics["model"] = "MLP_TFIDF_Reg"
    metrics["category"] = cat
    model3_results[cat] = metrics
    print(f"  RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, "
          f"Rounded Acc={metrics['rounded_acc']:.4f}")

pd.DataFrame(model3_results.values())

## Model 4: Multiclass Classification (5 classes)

Two sub-models:
- **LogisticRegression** (sklearn) on TF-IDF — linear baseline with class balancing
- **MLP** (PyTorch) on TF-IDF — 5-class output with CrossEntropyLoss

In [ ]:
# Model 4a: Logistic Regression (multiclass) on TF-IDF
model4a_results = {}
for cat in CATEGORIES:
    print(f"\n{cat}:")
    d = prepared[cat]
    model = LogisticRegression(
        multi_class="multinomial", class_weight="balanced",
        max_iter=300, solver="saga", C=1.0, n_jobs=-1
    )
    model.fit(d["X_tfidf_train"], d["y_class_train"])
    y_pred = model.predict(d["X_tfidf_test"])
    
    acc = accuracy_score(d["y_class_test"], y_pred)
    f1 = f1_score(d["y_class_test"], y_pred, average="macro")
    metrics = {"model": "LogReg_TFIDF_Cls", "category": cat, "accuracy": acc, "macro_f1": f1}
    model4a_results[cat] = metrics
    print(f"  Accuracy={acc:.4f}, Macro-F1={f1:.4f}")

pd.DataFrame(model4a_results.values())

In [ ]:
# Model 4b: MLP Classification (5 classes) on TF-IDF
model4b_results = {}
for cat in CATEGORIES:
    print(f"\n{'='*40} {cat}")
    d = prepared[cat]
    
    train_ds = SparseDataset(d["X_tfidf_train"], d["y_class_train"])
    test_ds = SparseDataset(d["X_tfidf_test"], d["y_class_test"])
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)
    
    input_dim = d["X_tfidf_train"].shape[1]
    model = RatingMLP(input_dim, output_dim=5)
    preds = train_mlp(model, train_loader, test_loader, epochs=10, task="classification")
    
    y_pred = preds.argmax(dim=1).numpy()
    acc = accuracy_score(d["y_class_test"], y_pred)
    f1 = f1_score(d["y_class_test"], y_pred, average="macro")
    metrics = {"model": "MLP_TFIDF_Cls", "category": cat, "accuracy": acc, "macro_f1": f1}
    model4b_results[cat] = metrics
    print(f"  Accuracy={acc:.4f}, Macro-F1={f1:.4f}")

pd.DataFrame(model4b_results.values())

## Comparison: All Models x All Categories

In [ ]:
# Build unified comparison table
all_results = []
for results in [model1_results, model2_results, model3_results]:
    all_results.extend(results.values())
for results in [model4a_results, model4b_results]:
    all_results.extend(results.values())

comparison_df = pd.DataFrame(all_results)
comparison_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("Model Comparison Summary")
comparison_df

In [ ]:
# Visualization: Accuracy comparison grouped by model and category
# For regression models use rounded_acc; for classification use accuracy
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Regression models — Rounded Accuracy
reg_df = comparison_df[comparison_df["model"].str.contains("Reg|Simple")].copy()
if not reg_df.empty:
    pivot_reg = reg_df.pivot(index="category", columns="model", values="rounded_acc")
    pivot_reg.plot(kind="bar", ax=axes[0], rot=20, alpha=0.85)
    axes[0].set_title("Regression Models — Rounded Accuracy")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend(loc="lower right")
    axes[0].set_ylim(0, 1)

# Plot 2: Classification models — Accuracy and Macro-F1
cls_df = comparison_df[comparison_df["model"].str.contains("Cls")].copy()
if not cls_df.empty:
    pivot_cls = cls_df.pivot(index="category", columns="model", values="accuracy")
    pivot_cls.plot(kind="bar", ax=axes[1], rot=20, alpha=0.85)
    axes[1].set_title("Classification Models — Accuracy")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend(loc="lower right")
    axes[1].set_ylim(0, 1)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices for the best classification model per category
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
rating_labels = ["1", "2", "3", "4", "5"]

for ax, cat in zip(axes.flat, CATEGORIES):
    d = prepared[cat]
    # Re-run LogReg prediction for confusion matrix (fast)
    model = LogisticRegression(
        multi_class="multinomial", class_weight="balanced",
        max_iter=300, solver="saga", C=1.0, n_jobs=-1
    )
    model.fit(d["X_tfidf_train"], d["y_class_train"])
    y_pred = model.predict(d["X_tfidf_test"])
    
    cm = confusion_matrix(d["y_class_test"], y_pred, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=rating_labels, yticklabels=rating_labels, ax=ax)
    ax.set_title(f"{cat}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.suptitle("Confusion Matrices — LogReg TF-IDF Multiclass (normalized)", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()

## Key Takeaways

1. **Text carries strong signal for rating prediction.** TF-IDF features significantly outperform simple word-count features, confirming that the *content* of reviews (not just their length) is informative.

2. **Linear models are competitive.** Ridge regression on TF-IDF is a strong baseline — the MLP provides marginal improvement, suggesting the text-to-rating relationship is roughly linear in TF-IDF space.

3. **Category difficulty varies.** Expect more polarized categories (e.g., Books with strong love/hate reviews) to be easier to predict than neutral-skewed categories.

4. **Class imbalance matters.** The heavy skew toward 5-star ratings means accuracy alone is misleading — Macro-F1 and confusion matrices reveal that low-star classes are harder to predict.

**Implications for the recommendation system:**
- Text-derived features (TF-IDF or embeddings) should be included in the ranking model's feature set
- Consider text encoding as a dense feature in the two-stage ranking pipeline
- The rating prediction task itself could serve as a pre-training objective for user/item embeddings